In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
# dtype={'zipCode': str} ensures zip codes like '02110' don't become 2110
df = pd.read_csv('HousingAssistanceOwners.csv', dtype={'zipCode': str})

# Standardize column names (optional, but good practice)
df.columns = [c.strip() for c in df.columns]

# Drop the unique row identifier as it's not needed for analysis
if 'id' in df.columns:
    df = df.drop(columns=['id'])

# Preview the clean dataframe
print(f"Loaded {len(df)} records.")
df.head()

Loaded 157918 records.


,disasterNumber,state,county,city,zipCode,validRegistrations,averageFemaInspectedDamage,totalInspected,totalDamage,noFemaInspectedDamage,...,femaInspectedDamageGreaterThan30000,approvedForFemaAssistance,totalApprovedIhpAmount,repairReplaceAmount,rentalAmount,otherNeedsAmount,approvedBetween1And10000,approvedBetween10001And25000,approvedBetween25001AndMax,totalMaxGrants
0,1439,TX,Aransas (County),ARANSAS PASS,78335,4,1345.01,3,5380.02,0,...,0,3,5915.91,3573.02,970.0,1372.89,3,0,0,0
1,1439,TX,Aransas (County),ARANSAS PASS,78336,68,3082.82,63,209632.06,4,...,0,54,180717.57,131097.36,23946.0,25674.21,27,26,1,0
2,1439,TX,Aransas (County),FULTON,78358,20,4722.34,18,94446.72,0,...,0,17,94240.88,58754.70,6784.0,28702.18,3,11,3,0
3,1439,TX,Aransas (County),FULTON,78381,1,2578.30,1,2578.30,0,...,0,1,3304.30,1407.96,726.0,1170.34,1,0,0,0
4,1439,TX,Aransas (County),ROCKPORT,78331,1,0.00,0,0.00,0,...,0,0,0.00,0.00,0.0,0.00,0,0,0,0


In [ ]:
# 1. Damage Intensity: Average damage per inspected home
# (Handling division by zero if totalInspected is 0)
df['avg_damage_per_inspection'] = np.where(
    df['totalInspected'] > 0,
    df['totalDamage'] / df['totalInspected'],
    0
)

# 2. Denial Rate: % of people who applied but got $0 (High denial rate might imply insurance covered it, or strictly minor damage)
df['denial_rate'] = 1 - (df['approvedForFemaAssistance'] / df['validRegistrations'])

# 3. Severity Ratio: The proportion of inspected homes with >$30k damage (Major catastrophic loss)
df['catastrophic_ratio'] = np.where(
    df['totalInspected'] > 0,
    df['femaInspectedDamageGreaterThan30000'] / df['totalInspected'],
    0
)

# 4. Financial Gap Proxy: Average shortfall per approved applicant
# (Note: This is a rough proxy as FEMA covers uninsured losses)
df['avg_grant_amount'] = np.where(
    df['approvedForFemaAssistance'] > 0,
    df['totalApprovedIhpAmount'] / df['approvedForFemaAssistance'],
    0
)

print(df[['zipCode', 'disasterNumber', 'avg_damage_per_inspection', 'catastrophic_ratio']].head())

  zipCode  disasterNumber  avg_damage_per_inspection  catastrophic_ratio
0   78335            1439                1793.340000                 0.0
1   78336            1439                3327.493016                 0.0
2   78358            1439                5247.040000                 0.0
3   78381            1439                2578.300000                 0.0
4   78331            1439                   0.000000                 0.0


In [ ]:
# Aggregate data by Zip Code to create a "Risk Profile" for each location
zip_risk_profile = df.groupby('zipCode').agg({
    'disasterNumber': 'nunique',              # Count of unique disasters hitting this zip
    'validRegistrations': 'sum',              # Total homeowners affected over time
    'totalDamage': 'sum',                     # Cumulative physical damage estimate
    'totalApprovedIhpAmount': 'sum',          # Total FEMA cash injected
    'catastrophic_ratio': 'mean'              # Average severity when hit
}).reset_index()

# Rename columns for clarity
zip_risk_profile.columns = [
    'zipCode', 'disaster_count', 'total_affected_homes',
    'cumulative_damage_usd', 'total_fema_payout_usd', 'avg_severity_score'
]

# Calculate a simplified "Risk Score" (e.g., Damage per impacted home normalized)
zip_risk_profile['risk_intensity'] = (
    zip_risk_profile['cumulative_damage_usd'] / zip_risk_profile['total_affected_homes']
).fillna(0)

# Sort by highest cumulative damage to see the riskiest zips
print("Top 10 Riskiest Zip Codes by Cumulative Damage:")
print(zip_risk_profile.sort_values(by='cumulative_damage_usd', ascending=False).head(10))

Top 10 Riskiest Zip Codes by Cumulative Damage:
      zipCode  disaster_count  total_affected_homes  cumulative_damage_usd  \
27673   91001               6                  9592           1.060642e+09   
27590   90272               5                  6657           7.057566e+08   
22660   70043              13                 16609           4.116786e+08   
22732   70126              12                 20638           3.175013e+08   
14124   39520               9                  8461           3.001135e+08   
11846   33908              10                 18217           2.803436e+08   
22728   70122              14                 27027           2.679201e+08   
22723   70117              12                 20523           2.466115e+08   
11867   33931              11                  8884           2.448884e+08   
22730   70124               8                 14906           2.414161e+08   

       total_fema_payout_usd  avg_severity_score  risk_intensity  
27673           2.577797e+